# Notebook 01 — Data Cleaning, Target Construction & Train/Test Split

**Proyecto:** Boston Marathon BQ Predictor  
**Autor:** Gian Marco  
**Fecha:** Abril 2026

## Objetivos

1. Cargar los 3 CSVs (Results, Races, BQStandards)
2. Aplicar criterios de limpieza documentados en `DECISIONS.md`
3. Construir la variable target `es_BQ`
4. Hacer merge con metadata de carreras (Races.csv)
5. Generar muestra estratificada de 300k
6. Aplicar split temporal: train 2022-2023 / test 2024
7. Guardar datasets procesados

## Referencia

Todas las decisiones aplicadas aquí están justificadas en `DECISIONS.md`.

---
## 1. Imports y configuración

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Reproducibilidad
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Rutas
RAW_DATA_DIR = Path('../data/raw')
PROCESSED_DATA_DIR = Path('../data/processed')
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Config pandas para ver todo
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

---
## 2. Carga de los 3 datasets

In [2]:
results = pd.read_csv(RAW_DATA_DIR / 'Results.csv', low_memory=False)
races = pd.read_csv(RAW_DATA_DIR / 'Races.csv')
bq = pd.read_csv(RAW_DATA_DIR / 'BQStandards.csv')

print(f'Results: {results.shape}')
print(f'Races:   {races.shape}')
print(f'BQ:      {bq.shape}')

Results: (1760979, 12)
Races:   (762, 8)
BQ:      (22, 3)


In [3]:
# Vista rápida
results.head(3)

,Year,Race,Name,Country,Zip,City,State,Gender,Age,Finish,Overall Place,Gender Place
0,2022,NYC Marathon,Evans Chebet,US,NaN,NaN,New York,M,33,7721.0,1,1.0
1,2022,NYC Marathon,Shura Kitata,US,NaN,NaN,New York,M,26,7734.0,2,2.0
2,2022,NYC Marathon,Abdi Nageeye,US,NaN,NaN,New York,M,33,7831.0,3,3.0


In [4]:
races.head(3)

,Year,Date,Race,City,State,Finishers,Include,Category
0,2022,2022-11-06,NYC Marathon,New York,NY,47738,Yes,NaN
1,2023,2023-11-05,NYC Marathon,New York,NY,51348,Yes,NaN
2,2022,2022-10-30,Marine Corps Marathon,Washington,DC,11257,Yes,NaN


In [5]:
bq.head(3)

,Gender,Age Bracket,Standard
0,M,Under 35,10800
1,M,35-39,11100
2,M,40-44,11400


---
## 3. Limpieza de Results.csv

Aplicamos los filtros documentados en `DECISIONS.md` Decisión 6. Llevamos el conteo de cuántos registros eliminamos en cada paso para tener trazabilidad.

In [6]:
df = results.copy()
cleaning_log = [('Inicial', len(df))]

# 3.1 Age a numérico (había valores con espacio en blanco)
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

# 3.2 Quitar DNF (Finish = 0)
df = df[df['Finish'] > 0]
cleaning_log.append(('Tras quitar DNF', len(df)))

# 3.3 Solo M y F (BQ Standards no tiene definidos otros géneros)
df = df[df['Gender'].isin(['M', 'F'])]
cleaning_log.append(('Tras filtrar M/F', len(df)))

# 3.4 Edades razonables (18-85)
df = df[(df['Age'] >= 18) & (df['Age'] <= 85)]
cleaning_log.append(('Tras filtrar edad válida', len(df)))

# 3.5 Drop columnas con muchos nulos o sin valor predictivo
df = df.drop(columns=['Zip', 'City', 'State', 'Name'])
cleaning_log.append(('Tras drop columnas irrelevantes', len(df)))

# Log visual
print('LOG DE LIMPIEZA DE RESULTS')
print('=' * 50)
for paso, n in cleaning_log:
    print(f'{paso:40s} {n:>12,}')

LOG DE LIMPIEZA DE RESULTS
Inicial                                     1,760,979
Tras quitar DNF                             1,746,679
Tras filtrar M/F                            1,722,190
Tras filtrar edad válida                    1,615,216
Tras drop columnas irrelevantes             1,615,216


### 3.6 Normalizar Country

Detectamos duplicados en códigos de país (`US`/`USA`, `GB`/`GBR`, `DE`/`GER`). Los unificamos.

In [7]:
# Mapping exhaustivo: TODOS los códigos no-ISO-2 → ISO 2-letra
country_mapping = {
    # ISO 3-letra estándar (comunes)
    'USA': 'US', 'GBR': 'GB', 'CAN': 'CA', 'FRA': 'FR', 'AUS': 'AU',
    'GER': 'DE', 'ITA': 'IT', 'IRL': 'IE', 'MEX': 'MX', 'RUS': 'RU',
    'ESP': 'ES', 'JPN': 'JP', 'ARG': 'AR', 'TWN': 'TW', 'DEN': 'DK',
    'SWE': 'SE', 'AUT': 'AT', 'BRA': 'BR', 'SIN': 'SG', 'BEL': 'BE',
    'HKG': 'HK', 'POL': 'PL', 'NZL': 'NZ', 'NOR': 'NO', 'SVK': 'SK',
    'POR': 'PT', 'CZE': 'CZ', 'IND': 'IN', 'EST': 'EE', 'FIN': 'FI',
    'CHN': 'CN', 'MAR': 'MA', 'TUR': 'TR', 'ISR': 'IL', 'PHI': 'PH',
    'THA': 'TH', 'COL': 'CO', 'ROU': 'RO', 'HUN': 'HU', 'KEN': 'KE',
    'LTU': 'LT', 'ISL': 'IS', 'UKR': 'UA', 'VIE': 'VN', 'KOR': 'KR',
    'PER': 'PE', 'VEN': 'VE', 'PAN': 'PA', 'TAN': 'TZ', 'ECU': 'EC',
    'DOM': 'DO', 'CRO': 'HR', 'ETH': 'ET', 'LUX': 'LU', 'EGY': 'EG',
    'UGA': 'UG', 'SRB': 'RS', 'BOL': 'BO',

    # ISO 3-letra menos comunes (países con pocos corredores)
    'MGL': 'MN', 'ERI': 'ER', 'GRE': 'GR', 'JOR': 'JO', 'CRC': 'CR',
    'IRI': 'IR', 'TRI': 'TT', 'KAZ': 'KZ', 'PUR': 'PR', 'LIE': 'LI',
    'ALG': 'DZ', 'LIB': 'LB', 'PAK': 'PK', 'BAN': 'BD', 'ZIM': 'ZW',
    'GEO': 'GE', 'GHA': 'GH', 'ANG': 'AO', 'AZE': 'AZ', 'NEP': 'NP',
    'BAH': 'BS', 'NCA': 'NI', 'TPE': 'TW', 'ARM': 'AM', 'KSA': 'SA',
    'KGZ': 'KG', 'LBA': 'LY', 'MLT': 'MT', 'MDA': 'MD', 'MON': 'MC',
    'SEN': 'SN', 'SRI': 'LK', 'TOG': 'TG', 'TRN': 'TT', 'TUN': 'TN',
    'BDI': 'BI', 'COD': 'CD', 'MAC': 'MO', 'CIV': 'CI', 'CAF': 'CF',
    'VIN': 'VC', 'MOZ': 'MZ', 'COM': 'KM', 'DMA': 'DM', 'DJI': 'DJ',

    # Códigos atípicos (no ISO estándar)
    'NED': 'NL',  # Netherlands
    'SUI': 'CH',  # Switzerland
    'INA': 'ID',  # Indonesia
    'RSA': 'ZA',  # South Africa
    'CHI': 'CL',  # Chile (ojo: NO es China, que sería CN)
    'BUR': 'UNKNOWN',  # Ambiguo: Burundi o Burkina Faso, solo 2 casos

    # Extra
    'LAT': 'LV', 'BRN': 'BH', 'MAS': 'MY', 'BUL': 'BG', 'FRO': 'FO',
    'PAR': 'PY', 'QAT': 'QA', 'SWZ': 'SZ', 'GUA': 'GT', 'KOS': 'XK',
    'ESA': 'SV', 'SLO': 'SI', 'BLR': 'BY', 'CYP': 'CY', 'BIH': 'BA',
    'MDV': 'MV', 'URU': 'UY', 'HON': 'HN', 'GIB': 'GI', 'MKD': 'MK',
    'ZAM': 'ZM', 'GRL': 'GL', 'SUD': 'SD', 'CHA': 'TD', 'BRU': 'BN',
    'AND': 'AD', 'IRQ': 'IQ', 'ALB': 'AL', 'MRI': 'MU', 'PLE': 'PS',
    'BER': 'BM', 'GUM': 'GU', 'NGR': 'NG', 'JAM': 'JM', 'CUB': 'CU',
    'GRN': 'GD', 'AFG': 'AF', 'HAI': 'HT', 'SUR': 'SR', 'LCA': 'LC',
    'NAM': 'NA', 'UZB': 'UZ', 'SYR': 'SY', 'MNE': 'ME', 'CMR': 'CM',
    'OMN': 'OM', 'TKM': 'TM', 'BEN': 'BJ', 'SMR': 'SM', 'PNG': 'PG',
    'KUW': 'KW', 'CPV': 'CV', 'BIZ': 'BZ', 'MAD': 'MG', 'GUY': 'GY',
    'UAE': 'AE', 'CAM': 'KH', 'BOT': 'BW', 'LAO': 'LA', 'PLW': 'PW',
    'RWA': 'RW', 'MAW': 'MW',

}

df['Country'] = df['Country'].replace(country_mapping)
df['Country'] = df['Country'].fillna('UNKNOWN')

# Verificación final
countries_3_letters = df[df['Country'].str.len() == 3]['Country'].unique()
if len(countries_3_letters) > 0:
    print(f'ADVERTENCIA: aún quedan códigos de 3 letras: {countries_3_letters}')
    print(f'Total de corredores afectados: {df[df["Country"].isin(countries_3_letters)].shape[0]}')
else:
    print('OK: todos los códigos son ISO-2 o UNKNOWN')

print(f'\nPaíses únicos tras normalizar: {df["Country"].nunique()}')
print(f'\nTop 15:')
print(df['Country'].value_counts().head(15))

OK: todos los códigos son ISO-2 o UNKNOWN

Países únicos tras normalizar: 216

Top 15:
Country
US         775536
GB         159970
ZA          56242
CA          54653
AU          50958
DE          47887
IT          42237
NL          37351
MX          36064
IE          35325
ES          31211
RU          28116
UNKNOWN     20562
JP          20159
FR          19220
Name: count, dtype: int64


---
## 4. Merge con Races.csv

Queremos traernos info de la carrera: `Category` (dificultad), `Finishers` (tamaño), y filtrar solo `Include == 'Yes'` para respetar el filtrado del autor.

⚠️ **Nota importante:** Antes del filtro `Include=Yes`, guardamos un snapshot con todas las carreras para poder construir el slice narrativo de España más adelante (las carreras españolas no tienen `Include=Yes` porque el autor se enfocó en USA/Canadá + London/Berlin).

In [8]:
# Guardar snapshot con todas las carreras (pre-filtro Include) para el slice España
df_all_races = df.copy()

# Ahora sí filtramos para el pipeline principal
races_clean = races[races['Include'] == 'Yes'].copy()
races_clean = races_clean[['Year', 'Race', 'City', 'State', 'Finishers', 'Category']]
races_clean = races_clean.rename(columns={'City': 'Race_City', 'State': 'Race_State'})

n_before = len(df)
df = df.merge(races_clean, on=['Year', 'Race'], how='inner')
print(f'Filas tras merge (solo carreras Include=Yes): {len(df):,}')
print(f'Filas eliminadas: {n_before - len(df):,}')

cleaning_log.append(('Tras merge con Races (Include=Yes)', len(df)))

Filas tras merge (solo carreras Include=Yes): 1,068,226
Filas eliminadas: 546,990


---
## 5. Construcción del target `es_BQ`

Asignamos a cada corredor su franja de edad según BQ Standards, hacemos merge y creamos la variable target.

In [9]:
def assign_age_bracket(age):
    """Asigna la franja de edad según categorías oficiales BQ."""
    if age < 35: return 'Under 35'
    elif age < 40: return '35-39'
    elif age < 45: return '40-44'
    elif age < 50: return '45-49'
    elif age < 55: return '50-54'
    elif age < 60: return '55-59'
    elif age < 65: return '60-64'
    elif age < 70: return '65-69'
    elif age < 75: return '70-74'
    elif age < 80: return '75-79'
    else: return '80 and Over'

df['Age Bracket'] = df['Age'].apply(assign_age_bracket)

# Merge con BQ para traer Standard
df = df.merge(bq, on=['Gender', 'Age Bracket'], how='left')

# Crear target
df['es_BQ'] = (df['Finish'] <= df['Standard']).astype(int)

print('Distribución del target:')
print(df['es_BQ'].value_counts())
print(f'\n% BQ: {df["es_BQ"].mean()*100:.2f}%')
print(f'Ratio desbalance: 1:{(df["es_BQ"]==0).sum() / (df["es_BQ"]==1).sum():.2f}')

Distribución del target:
es_BQ
0    922271
1    145955
Name: count, dtype: int64

% BQ: 13.66%
Ratio desbalance: 1:6.32


---
## 6. Sanity checks

Antes de muestrear, verificamos que los datos tienen sentido.

In [10]:
print('% BQ por año (debe ir creciendo según nuestra observación previa):')
print(df.groupby('Year')['es_BQ'].agg(['mean', 'sum', 'count']).round(3))

print('\n% BQ por género:')
print(df.groupby('Gender')['es_BQ'].agg(['mean', 'count']).round(3))

print('\nDistribución de carreras (top 10):')
print(df['Race'].value_counts().head(10))

% BQ por año (debe ir creciendo según nuestra observación previa):
       mean    sum   count
Year                      
2022  0.117  33659  286548
2023  0.144  74276  515888
2024  0.143  38020  265790

% BQ por género:
         mean   count
Gender               
F       0.141  424447
M       0.134  643779

Distribución de carreras (top 10):
Race
London Marathon          143144
NYC Marathon              98934
Chicago Marathon          87211
Berlin Marathon           77742
Boston Marathon           52099
LA Marathon               32743
Honolulu Marathon         28548
Disney World Marathon     25402
Marine Corps Marathon     24757
Philadelphia Marathon     19672
Name: count, dtype: int64


---
## 7. Muestreo estratificado

**Estrategia:** Muestrear 300k filas estratificando por `es_BQ × Year × Gender` para preservar las distribuciones críticas (ver `DECISIONS.md` Decisión 4).

In [11]:
TARGET_SAMPLE_SIZE = 300_000

# Calcular la fracción
frac = TARGET_SAMPLE_SIZE / len(df)
print(f'Fracción a muestrear: {frac:.4f}')

# Estratificación por múltiples columnas combinadas
strata = df['es_BQ'].astype(str) + '_' + df['Year'].astype(str) + '_' + df['Gender']

df_sample = (
    df.groupby(strata, group_keys=False)
      .apply(lambda g: g.sample(frac=frac, random_state=RANDOM_STATE))
      .reset_index(drop=True)
)

print(f'Tamaño de la muestra: {len(df_sample):,}')
print(f'\n% BQ en muestra: {df_sample["es_BQ"].mean()*100:.2f}% (objetivo: ~12.76%)')

print('\n% BQ por año en muestra (debe parecerse al original):')
print(df_sample.groupby('Year')['es_BQ'].agg(['mean', 'count']).round(3))

Fracción a muestrear: 0.2808
Tamaño de la muestra: 300,000

% BQ en muestra: 13.66% (objetivo: ~12.76%)

% BQ por año en muestra (debe parecerse al original):
       mean   count
Year               
2022  0.117   80475
2023  0.144  144881
2024  0.143   74644


---
## 8. Split temporal

**Train:** 2022-2023
**Test:** 2024

Justificación completa en `DECISIONS.md` Decisión 3.

In [12]:
train = df_sample[df_sample['Year'].isin([2022, 2023])].reset_index(drop=True)
test = df_sample[df_sample['Year'] == 2024].reset_index(drop=True)

print(f'Train: {len(train):,} filas  |  % BQ: {train["es_BQ"].mean()*100:.2f}%')
print(f'Test:  {len(test):,} filas  |  % BQ: {test["es_BQ"].mean()*100:.2f}%')
print(f'\nRatio train/test: {len(train)/len(test):.2f}')

Train: 225,356 filas  |  % BQ: 13.45%
Test:  74,644 filas  |  % BQ: 14.30%

Ratio train/test: 3.02


---
## 9. Slice narrativo: España

Reservamos los registros de carreras españolas para análisis posterior (notebook 09) y la demo de Streamlit.

In [13]:
# Aplicamos el mismo proceso de target a df_all_races (pre-filtro Include)
df_all_races['Age Bracket'] = df_all_races['Age'].apply(assign_age_bracket)
df_all_races = df_all_races.merge(bq, on=['Gender', 'Age Bracket'], how='left')
df_all_races['es_BQ'] = (df_all_races['Finish'] <= df_all_races['Standard']).astype(int)

# Identificar carreras celebradas en España
spain_keywords = ['Madrid', 'Barcelona', 'Sevilla', 'Valencia', 'Lanzarote']
spain_races_mask = df_all_races['Race'].str.contains('|'.join(spain_keywords), case=False, na=False)

# 'Zurich Marato' (catalán/español) para Barcelona/Sevilla
zurich_spain = df_all_races['Race'].str.contains('Zurich Marato', case=False, na=False)
spain_races_mask = spain_races_mask | zurich_spain

spain_slice = df_all_races[spain_races_mask].reset_index(drop=True)

print(f'Slice España: {len(spain_slice):,} filas')
print(f'\nCarreras españolas incluidas:')
print(spain_slice.groupby(['Race', 'Year']).agg(
    finishers=('es_BQ', 'count'),
    bq_pct=('es_BQ', 'mean')
).round(3))

Slice España: 31,419 filas

Carreras españolas incluidas:
                              finishers  bq_pct
Race                    Year                   
RnR Madrid Marathon     2023       7387   0.107
                        2024       9059   0.107
Zurich Marato Barcelona 2023       7917   0.162
Zurich Maraton Sevilla  2024       7056   0.316


---
## 10. Guardar datasets procesados

In [14]:
# Definir rutas específicas
TRAIN_DIR = Path('../data/train')
TEST_DIR = Path('../data/test')

# Guardar datos limpios en sus respectivas carpetas
train.to_csv(TRAIN_DIR / 'train.csv', index=False)
test.to_csv(TEST_DIR / 'test.csv', index=False)
spain_slice.to_csv(TEST_DIR / 'spain_slice.csv', index=False)

# El cleaning_log es metadatos del pipeline, va a processed
cleaning_df = pd.DataFrame(cleaning_log, columns=['Paso', 'N_filas'])
cleaning_df.to_csv(PROCESSED_DATA_DIR / 'cleaning_log.csv', index=False)

print('Archivos guardados:')
print(f'\ndata/train/:')
for f in sorted(TRAIN_DIR.iterdir()):
    if f.name != '.gitkeep':
        size_mb = f.stat().st_size / 1024**2
        print(f'  {f.name:25s} {size_mb:>6.1f} MB')

print(f'\ndata/test/:')
for f in sorted(TEST_DIR.iterdir()):
    if f.name != '.gitkeep':
        size_mb = f.stat().st_size / 1024**2
        print(f'  {f.name:25s} {size_mb:>6.1f} MB')

print(f'\ndata/processed/:')
for f in sorted(PROCESSED_DATA_DIR.iterdir()):
    if f.name != '.gitkeep':
        size_mb = f.stat().st_size / 1024**2
        print(f'  {f.name:25s} {size_mb:>6.1f} MB')

Archivos guardados:

data/train/:
  train.csv                   19.0 MB
  train_features.csv          16.9 MB

data/test/:
  spain_slice.csv              2.1 MB
  test.csv                     6.3 MB
  test_features.csv            5.6 MB

data/processed/:
  advanced_results.csv         0.0 MB
  baseline_results.csv         0.0 MB
  cleaning_log.csv             0.0 MB


---
## Resumen del notebook

| Paso | Resultado |
|---|---|
| Datos originales | 1.76M filas |
| Tras limpieza | ~1.6M filas |
| Tras merge con Races (Include=Yes) | ~1.5M filas (estimado) |
| Muestra estratificada | 300k filas |
| Train (2022-2023) | ~265k filas |
| Test (2024) | ~35k filas |
| Slice España | ~6k filas |
| **Target % BQ** | **~12.76%** |